# IMC 2025: DINOv2 retrieval + ALIKED/LightGlue + COLMAP

Groups each dataset's images into scenes and estimates camera poses.
Code and write-up: see the GitHub repo (`kaggle-image-matching`). Internet is off; wheels, weights
and the `imc` package come from the attached `imc25-offline-deps` dataset.

In [ ]:
import glob, os, shutil, subprocess, sys
DEPS = '/kaggle/input/imc25-offline-deps'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', '--no-deps',
                '--find-links', f'{DEPS}/wheels', 'pycolmap', 'lightglue', 'kornia', 'kornia_rs'], check=False)
# LightGlue/ALIKED load weights through the torch hub cache.
hub = os.path.expanduser('~/.cache/torch/hub/checkpoints'); os.makedirs(hub, exist_ok=True)
for f in glob.glob(f'{DEPS}/weights/*.pth'):
    shutil.copy(f, hub)
# Copy the package somewhere writable; the SfM stage runs as `python -m imc.sfm` from there.
shutil.copytree(f'{DEPS}/src/imc', '/kaggle/working/imc', dirs_exist_ok=True)
os.chdir('/kaggle/working'); sys.path.insert(0, '/kaggle/working')

In [ ]:
import time, traceback
from pathlib import Path
import pandas as pd
import torch
from imc.data import NAN_R, NAN_T, test_datasets, write_submission
from imc.features import FeatureConfig
from imc.pipeline import run_dataset

DATA = Path('/kaggle/input/image-matching-challenge-2025-ongoing')
cfg = FeatureConfig(dino_model=f'{DEPS}/weights/dinov2-base')
print(torch.__version__, torch.cuda.is_available())
datasets = test_datasets(DATA)
print({d: len(v) for d, v in datasets.items()})

In [ ]:
BUDGET_S = 8 * 3600  # Kaggle limit is 9 h; leave room to write the file
start, rows = time.time(), []
for d, names in datasets.items():
    if time.time() - start > BUDGET_S:
        print('time budget hit, marking', d, 'as outliers')
        rows += [dict(dataset=d, scene='outliers', image=n, rotation_matrix=NAN_R, translation_vector=NAN_T) for n in names]
        continue
    try:
        r, stats = run_dataset(d, DATA / 'test' / d, names, Path('/tmp/imc_work'), cfg)
        rows += r
        print(d, {k: v for k, v in stats.items() if k != 'sfm_error'})
        if 'sfm_error' in stats: print(stats['sfm_error'][-500:])
    except Exception:
        traceback.print_exc()
        rows += [dict(dataset=d, scene='outliers', image=n, rotation_matrix=NAN_R, translation_vector=NAN_T) for n in names]
sub = write_submission(rows, Path('/kaggle/working/submission.csv'), DATA / 'sample_submission.csv')
shutil.rmtree('/kaggle/working/imc', ignore_errors=True)
print(sub.shape, f'{(time.time() - start) / 60:.1f} min'); sub.head()